In [4]:
# Expérience 1 (15 figures + la figure bonus et le tableau résumé)
# Libraires Python 
import os #os : gestion des fichiers/dossiers (créer sorties_experience1).
import numpy as np # calculs numériques.
import pandas as pd  # gestion des séries temporelles (index par temps).
import xarray as xr # lecture et manipulation du fichier netCDF.
import matplotlib.pyplot as plt   #création de graphiques.
import matplotlib.dates as mdates
import cartopy.crs as ccrs  #cartes géographiques (type de projection, côtes, frontières)
import cartopy.feature as cfeature
from scipy.stats import pearsonr # pour calculer le coefficient de corrélation de Pearson

# la configuration, préparation des données
chemin_fichier = "/Users/PC/Downloads/imerg_pr_201911_3h.nc4"
nom_variable = "precipitationCal"   # mm (3h)-1

emplacements = {
    "Kuala_Lumpur": (3.1, 101.6),
    "Montreal": (45.5, -73.5),
    "Ocean_point": (5.0, 106.0),
    "Alter_do_Chao": (-2.5, -54.95)
}

dossier_sortie = "sorties_figures_tableau_experience_1"# Création du dossier où sauvegarder les figures et tableau 
os.makedirs(dossier_sortie, exist_ok=True)

# ========== UTILITAIRES ==========
def indice_plus_proche(array, valeur):# Trouve l’indice de la grille le plus proche d’une latitude/longitude donnée.
    return int(np.abs(array - valeur).argmin())

def degres_offset_50km(lat):#calcule combien de degrés correspondent à ~50 km (selon latitude).
    deg_lat = 50.0 / 111.0
    deg_lon = 50.0 / (111.0 * np.cos(np.deg2rad(lat)))
    return deg_lat, deg_lon

def trouver_voisins(latitudes, longitudes, lat0, lon0):#retourne les indices du point central + ceux qui sont voisins N/S/E/O à ~50 km
    i0 = indice_plus_proche(latitudes, lat0)
    j0 = indice_plus_proche(longitudes, lon0)
    dlat_deg, dlon_deg = degres_offset_50km(lat0)
    i_nord = indice_plus_proche(latitudes, latitudes[i0] + dlat_deg)
    i_sud  = indice_plus_proche(latitudes, latitudes[i0] - dlat_deg)
    j_est  = indice_plus_proche(longitudes, longitudes[j0] + dlon_deg)
    j_ouest= indice_plus_proche(longitudes, longitudes[j0] - dlon_deg)
    return (i0, j0), (i_nord, j0), (i_sud, j0), (i0, j_est), (i0, j_ouest)

def durees_evenements_precipitation(serie):#calcule la durée des événements de pluie consécutifs (pas de 3h).
    masque = serie > 0
    if masque.sum() == 0:
        return []
    durees, longueur_courante = [], 0
    for m in masque:
        if m:
            longueur_courante += 3
        else:
            if longueur_courante > 0:
                durees.append(longueur_courante)
                longueur_courante = 0
    if longueur_courante > 0:
        durees.append(longueur_courante)
    return durees

# Lecture
print("Ouverture du fichier :", chemin_fichier)
ds = xr.open_dataset(chemin_fichier, chunks={'time': 1}) #Ouvre le fichier netCDF (avec dask pour gérer mémoire).
print(ds)

precipitations = ds[nom_variable] #Charge la variable de précipitation.
latitudes = ds['lat'].values #Extrait coordonnées latitude/longitude.
longitudes = ds['lon'].values

if longitudes.max() > 180:#Si longitudes en 0–360°, les convertit en -180–180
    longitudes = ((longitudes + 180) % 360) - 180
    precipitations = precipitations.assign_coords(lon=longitudes)
    precipitations = precipitations.sortby('lon')

# Analyse
#Pour un site donné cette partie du script extrait la série temporelle de précipitation pour calculer 
#l’accumulation totale, le nb de pas de pluie, fréquence, la moyenne globale et moyenne_quand_precip_>0, 
#le maximum, durée max d’un événement.
def analyser_emplacement(nom, lat0, lon0):
    lat_idx = indice_plus_proche(latitudes, lat0)
    lon_idx = indice_plus_proche(longitudes, lon0)
    ts = precipitations[:, lat_idx, lon_idx].compute().values
    accumulation_totale = np.nansum(ts)
    pluie = (ts > 0)
    nb_pluie = np.nansum(pluie)
    freq_pluie = nb_pluie / ts.size
    moyenne_tous = np.nanmean(ts)
    moyenne_quand_pluie = np.nanmean(ts[pluie]) if nb_pluie > 0 else np.nan
    max_taux = np.nanmax(ts)
    durees = durees_evenements_precipitation(ts)
    duree_max_evenement = max(durees) if durees else 0.0

    (i0, j0), nord, sud, est, ouest = trouver_voisins(latitudes, longitudes, latitudes[lat_idx], longitudes[lon_idx])#Cherche les points voisins N/S/E/O du point central et calcule corrélation avec eux.
    voisins_idx = [nord, sud, est, ouest]
    corr_voisins = {}
    for (ii, jj), etiquette in zip(voisins_idx, ['Nord','Sud','Est','Ouest']):
        voisin_ts = precipitations[:, ii, jj].compute().values
        masque = ~np.isnan(ts) & ~np.isnan(voisin_ts)
        if masque.sum() > 1:
            r, _ = pearsonr(ts[masque], voisin_ts[masque])
        else:
            r = np.nan
        corr_voisins[etiquette] = r

    index_temps = pd.to_datetime(ds['time'].values)
    df_ts = pd.DataFrame({'precipitations': ts}, index=index_temps)
    #Retourne toutes ces infos dans un dictionnaire.

    return {
        'nom': nom,
        'lat': latitudes[lat_idx],
        'lon': longitudes[lon_idx],
        'ts': ts,
        'index_temps': index_temps,
        'accumulation_totale': accumulation_totale,
        'nb_pluie': int(nb_pluie),
        'freq_pluie': freq_pluie,
        'moyenne_tous': moyenne_tous,
        'moyenne_quand_pluie': moyenne_quand_pluie,
        'max_taux': max_taux,
        'duree_max_evenement_h': duree_max_evenement,
        'corr_voisins': corr_voisins,
        'voisins_idx': voisins_idx,
        'df_ts': df_ts
    }

resultats = {nom: analyser_emplacement(nom, lat, lon) for nom, (lat, lon) in emplacements.items()}#Analyse faite pour les 4 emplacements.

# Tableau
lignes = []
for nom, res in resultats.items():
    lignes.append({
        'Emplacement': nom,
        'Latitude': res['lat'],
        'Longitude': res['lon'],
        'Accumulation_totale_mm': res['accumulation_totale'],
        'Nb_pas_pluie': res['nb_pluie'],
        'Fraction_temps_pluie': res['freq_pluie'],
        'Moyenne_tous_pas': res['moyenne_tous'],
        'Moyenne_quand_pluie': res['moyenne_quand_pluie'],
        'Max_mm_par_3h': res['max_taux'],
        'Duree_max_evenement_h': res['duree_max_evenement_h'],
        'Corr_Nord': res['corr_voisins']['Nord'],
        'Corr_Sud': res['corr_voisins']['Sud'],
        'Corr_Est': res['corr_voisins']['Est'],
        'Corr_Ouest': res['corr_voisins']['Ouest']
    })
df_resume = pd.DataFrame(lignes).set_index('Emplacement')#Construit un DataFrame avec toutes les stats pour chaque site
df_resume.to_csv(os.path.join(dossier_sortie, 'tableau_expérience_1.csv'))#Sauvegarde en CSV

# 1 ère figure Séries temporelles (4 sous-graphes pour les 4 emplacements)
plt.figure(figsize=(12,8))  # crée une figure globale
ymax = max([np.nanmax(res['ts']) for res in resultats.values()]) * 1.05  # valeur max des précipitations (pour homogénéiser l’échelle Y)
ymin = 0.0  # valeur minimale fixée à zéro

for i, (nom, res) in enumerate(resultats.items(), 1):  # boucle sur les emplacements
    ax = plt.subplot(2,2,i)  # crée un sous-graphe
    ax.plot(res['index_temps'], res['ts'])  # trace la série temporelle
    ax.set_title(f"{nom} - novembre 2019")
    ax.set_ylim(ymin, ymax)  # même échelle Y pour tous
    ax.set_ylabel("Précipitations (mm / 3h)")
    ax.set_xlabel("Jour")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d"))  # affiche seulement le jour sur l’axe X
plt.tight_layout()
plt.savefig(os.path.join(dossier_sortie, 'series_temporelles.png'), dpi=150)  # sauvegarde la figure
plt.close()

# 2ème figure Scatter plots (centre vs voisins N/S/E/O) 
for nom, res in resultats.items():  # boucle sur les emplacements
    ts_centre = res['ts']  # série du point central
    voisins_idx = res['voisins_idx']  # indices des voisins
    etiquettes = ['Nord','Sud','Est','Ouest']
    plt.figure(figsize=(10,8))
    for k, ((ii,jj), lab) in enumerate(zip(voisins_idx, etiquettes),1):
        voisin_ts = precipitations[:, ii, jj].compute().values  # série du voisin
        masque = ~np.isnan(ts_centre) & ~np.isnan(voisin_ts)  # enlève les valeurs manquantes
        plt.subplot(2,2,k)
        plt.scatter(ts_centre[masque], voisin_ts[masque], s=5)  # nuage de points
        plt.xlabel("Centre (mm/3h)")
        plt.ylabel(f"{lab} (mm/3h)")
        plt.title(f"{nom} vs {lab} (r={res['corr_voisins'][lab]:.2f})")  # affiche corrélation dans le titre
    plt.tight_layout()
    plt.savefig(os.path.join(dossier_sortie, f'Nuages_de_points_{nom}.png'), dpi=150)
    plt.close()

# 3ème figure Cycles diurnes moyens (0h, 3h, ..., 21h heure locale)
for nom, res in resultats.items():
    df = res['df_ts'].copy()
    lon_site = res['lon']
    heures_utc = df.index.hour  # heures UTC
    heures_locales = (heures_utc + lon_site/15.0) % 24  # conversion en heures locales
    heures_locales = (np.floor(heures_locales/3)*3).astype(int)  # arrondi aux pas de 3h
    df['heure_locale'] = heures_locales
    cycle = df.groupby('heure_locale')['precipitations'].mean()  # moyenne par heure locale
    cycle = cycle.reindex(np.arange(0,24,3))  # force l’ordre 0–3–...–21
    plt.plot(np.arange(0,24,3), cycle.values, marker='o')
    plt.title(f"Cycle diurne moyen novembre 2019 - {nom}")
    plt.xlabel("Heure locale")
    plt.ylabel("Précipitations (mm/3h)")
    plt.ylim(ymin, ymax)
    plt.xticks(np.arange(0,24,3))
    plt.grid(True)
    plt.savefig(os.path.join(dossier_sortie, f'cycle_diurne_{nom}.png'))
    plt.close()

# 4ème figure Carte globale pour le 1er novembre 2019 à 12 UTC
t_sel = np.datetime64('2019-11-01T12:00:00')
t_idx = int(np.abs(ds['time'].values - t_sel).argmin())  # trouve l’indice temporel le plus proche
champ = precipitations[t_idx,:,:].compute().values
champ_limite = np.where((champ>=0) & (champ<=20), champ, np.nan)  # limite entre 0 et 20 mm/3h
lon2d, lat2d = np.meshgrid(longitudes, latitudes)

plt.figure(figsize=(12,6))
ax = plt.axes(projection=ccrs.PlateCarree())
pcm = ax.pcolormesh(lon2d, lat2d, champ_limite, transform=ccrs.PlateCarree())#
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.set_title("Taux de précipitations (0–20 mm/3h) le 1 nov. 2019 12UTC")
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False; gl.right_labels = False
plt.colorbar(pcm, label='mm / 3h')
plt.savefig(os.path.join(dossier_sortie, 'carte_globale_1nov2019.png'), dpi=150)
plt.close()

# 5ème figure Histogramme des précipitations le 1er novembre 2019 à 12 UTC
valeurs = champ.flatten()
valeurs = valeurs[~np.isnan(valeurs)]
plt.figure()
plt.hist(valeurs, bins=20)  # histogramme avec 20 classes
plt.yscale('log')  # échelle logarithmique sur Y
plt.xlabel('Intensité des précipitations (mm/3h)')
plt.ylabel('Fréquence (log)')
plt.title('Histogramme frequence vs intensité 1 nov. 2019 12UTC')
plt.savefig(os.path.join(dossier_sortie, 'histogramme_1nov2019.png'), dpi=150)
plt.close()

# 6ème figure Carte de la moyenne mensuelle globale (novembre 2019)
moyenne_mensuelle = precipitations.mean(dim='time').compute().values
moyenne_limite = np.where((moyenne_mensuelle>=0) & (moyenne_mensuelle<=5), moyenne_mensuelle, np.nan)

plt.figure(figsize=(12,6))
ax = plt.axes(projection=ccrs.PlateCarree())
pcm = ax.pcolormesh(lon2d, lat2d, moyenne_limite, transform=ccrs.PlateCarree())
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.set_title("Intensité moyenne des précipitations sur nov. 2019 ")
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False; gl.right_labels = False
plt.colorbar(pcm, label='mm / 3h')
plt.savefig(os.path.join(dossier_sortie, 'carte_globale_moyenne_sur_novembre_2019.png'), dpi=150)
plt.close()

# 7ème Histogramme de la moyenne mensuelle (novembre 2019)
valeurs_m = moyenne_mensuelle.flatten() # aplatit le champ spatial
valeurs_m = valeurs_m[~np.isnan(valeurs_m)] # enlève les valeurs NaN
plt.figure()
plt.hist(valeurs_m, bins=20)
plt.yscale('log')# histogramme sous forme logarithmique 
plt.xlabel(' Intensité moyenne des précipitations(mm/3h)')
plt.ylabel('Fréquence (log)')
plt.title('Histogramme fréquence vs intensité moyenne sur nov. 2019')
plt.savefig(os.path.join(dossier_sortie, 'histogramme_moyenne_mensuelle.png'), dpi=150)
plt.close()

# 8ème Histogramme des précipitations (échelle linéaire)

valeurs_m = moyenne_mensuelle.flatten()          # aplatit le champ spatial
valeurs_m = valeurs_m[~np.isnan(valeurs_m)]      # enlève les valeurs NaN

plt.figure()
plt.hist(valeurs_m, bins=20, color="skyblue", edgecolor="black")  # histogramme classique
plt.xlabel('Précipitations moyennes (mm/3h)')
plt.ylabel('Fréquence')
plt.title('Histogramme fréquence (linéaire) vs intensité moyenne sur nov. 2019')
plt.grid(True, linestyle='--', alpha=0.6)        # ajoute une grille légère
plt.savefig(os.path.join(dossier_sortie, 'histogramme_moyenne_mensuelle_lineaire.png'), dpi=150)
plt.close()


# 9ème Zoom Montréal (fenêtre 2x2° autour du site)
lat_c, lon_c = emplacements['Montreal']
lat_min, lat_max = lat_c - 1.0, lat_c + 1.0
lon_min, lon_max = lon_c - 1.0, lon_c + 1.0
ilat_min = indice_plus_proche(latitudes, lat_min)
ilat_max = indice_plus_proche(latitudes, lat_max)
ilon_min = indice_plus_proche(longitudes, lon_min)
ilon_max = indice_plus_proche(longitudes, lon_max)
lat_sub = latitudes[ilat_min:ilat_max+1]
lon_sub = longitudes[ilon_min:ilon_max+1]
lon2d_sub, lat2d_sub = np.meshgrid(lon_sub, lat_sub)
moyenne_sub = moyenne_mensuelle[ilat_min:ilat_max+1, ilon_min:ilon_max+1]

# Carte pcolormesh
plt.figure(figsize=(6,6))
ax = plt.axes(projection=ccrs.Mercator())
pcm = ax.pcolormesh(lon2d_sub, lat2d_sub, moyenne_sub, transform=ccrs.PlateCarree())
ax.coastlines(resolution='10m')
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
ax.set_title("Montréal (2x2°) - taux moyen des precipitations (pcolormesh)")
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False; gl.right_labels = False
plt.colorbar(pcm, label='mm/3h')
plt.savefig(os.path.join(dossier_sortie, 'montreal_2deg_pcolormesh.png'), dpi=150)
plt.close()

# Carte contourf
plt.figure(figsize=(6,6))
ax = plt.axes(projection=ccrs.Mercator())
cf = ax.contourf(lon2d_sub, lat2d_sub, moyenne_sub, transform=ccrs.PlateCarree(), levels=10)
ax.coastlines(resolution='10m')
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
ax.set_title("Montréal (2x2°) - taux moyen des precipitations nov.2019 (contourf)")
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False; gl.right_labels = False
plt.colorbar(cf, label='mm/3h')
plt.savefig(os.path.join(dossier_sortie, 'montreal_2deg_contourf.png'), dpi=150)
plt.close()



    
def analyser_emplacement_bonus(lat0, lon0, precipitations, latitudes, longitudes):
    lat_idx = indice_plus_proche(latitudes, lat0)
    lon_idx = indice_plus_proche(longitudes, lon0)
    ts = precipitations[:, lat_idx, lon_idx].compute().values
    
    # moyenne_quand_precip_>0
    masque = ts > 0
    moyenne_quand_pluie = np.nanmean(ts[masque]) if masque.sum() > 0 else np.nan

    # corrélations avec voisins
    (i0, j0), nord, sud, est, ouest = trouver_voisins(latitudes, longitudes, latitudes[lat_idx], longitudes[lon_idx])
    voisins_idx = [nord, sud, est, ouest]
    corr_voisins = {}
    for (ii, jj), etiquette in zip(voisins_idx, ['Nord','Sud','Est','Ouest']):
        voisin_ts = precipitations[:, ii, jj].compute().values
        masque_valide = ~np.isnan(ts) & ~np.isnan(voisin_ts)
        if masque_valide.sum() > 1:
            r, _ = pearsonr(ts[masque_valide], voisin_ts[masque_valide])
        else:
            r = np.nan
        corr_voisins[etiquette] = r
    return corr_voisins, moyenne_quand_pluie

latitudes_ligne = np.linspace(60, -60, 20)
lon_ref = -73.5
valeurs_corr = []
valeurs_moy_pluie = []

for latv in latitudes_ligne:
    corr_dict, moy_pluie = analyser_emplacement_bonus(latv, lon_ref, precipitations, latitudes, longitudes)
    corr_moy = np.nanmean(list(corr_dict.values()))
    valeurs_corr.append(corr_moy)
    valeurs_moy_pluie.append(moy_pluie)

fig, ax1 = plt.subplots(figsize=(9,6))

# Axe 1 corrélation moyenne
ax1.plot(latitudes_ligne, valeurs_corr, marker='o', color='black', label="Corrélation moyenne (r)")
ax1.axhline(0.8, color='red', linestyle='--', label="Seuil forte corrélation (r=0.8)")
ax1.set_xlabel("Latitude (°)")
ax1.set_ylabel("Corrélation moyenne (r)", color='black')
ax1.tick_params(axis='y', labelcolor='black')
ax1.grid(True)

# Axe 2 moyenne quand precipipitations >0
ax2 = ax1.twinx()
ax2.bar(latitudes_ligne, valeurs_moy_pluie, width=3, alpha=0.4, color='green', label="moyenne_quand_precip_>0 (mm/3h)")
ax2.set_ylabel("moyenne_quand_precip_>0 (mm/3h)", color='green')
ax2.tick_params(axis='y', labelcolor='green')

# Légende combinée
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc="upper right")

plt.title("Corrélation moyenne des quatre points N/S/E/O et l'intensité moyenne des pluies\nsur le méridien de Montréal (-73.5°)")
plt.savefig(os.path.join(dossier_sortie, 'figure_bonus.png'), dpi=150)
plt.close()

print("Les 16 figures et le tableau sont dans", dossier_sortie)


Ouverture du fichier : /Users/PC/Downloads/imerg_pr_201911_3h.nc4
<xarray.Dataset> Size: 6GB
Dimensions:           (time: 240, bnds: 2, lon: 3600, lat: 1800)
Coordinates:
  * time              (time) datetime64[ns] 2kB 2019-11-01 ... 2019-11-30T21:...
  * lon               (lon) float32 14kB -179.9 -179.9 -179.8 ... 179.9 179.9
  * lat               (lat) float32 7kB -89.95 -89.85 -89.75 ... 89.85 89.95
Dimensions without coordinates: bnds
Data variables:
    time_bnds         (time, bnds) datetime64[ns] 4kB dask.array<chunksize=(1, 2), meta=np.ndarray>
    precipitationCal  (time, lat, lon) float32 6GB dask.array<chunksize=(1, 1800, 3600), meta=np.ndarray>
Attributes:
    CDI:          Climate Data Interface version 1.9.5 (http://mpimet.mpg.de/...
    history:      Mon Sep 27 17:58:00 2021: cdo --timestat_date first -L -f n...
    Conventions:  CF-1.6
    FileHeader:   DOI=10.5067/GPM/IMERG/3B-HH/06;\nDOIauthority=http://dx.doi...
    FileInfo:     DataFormatVersion=6a;\nTKCodeBuildVe